# COAWST Curvilinear Grid — WebGL Browser Demo

This notebook shows the **Python-side equivalent** of what the browser demo does:
open the COAWST icechunk store, read a time slice of a 2D variable, and visualize
it as a curvilinear quad mesh.

The browser demo (`npm run dev` → http://localhost:5173) reads the same data
directly from S3 using icechunk.js + zarrita, tessellates with
`curvilinear_mesh_layer.js`, and renders with deck.gl — no server needed.

Sample time: **2012-10-29 12:00 UTC** — Hurricane Sandy at peak intensity.

In [ ]:
import numpy as np
import xarray as xr
import hvplot.xarray
import icechunk
from icechunk import (
    ManifestConfig,
    ManifestSplitCondition,
    ManifestSplitDimCondition,
    ManifestSplittingConfig,
)

## 1. Open the COAWST icechunk store

This is identical to `COAWST_explore_icechunk.ipynb`.

In [ ]:
bucket = 'usgs-coawst'
region = 'us-west-2'
prefix = 'useast-archive/icechunk/coawst-useast.icechunk'
TIME_DIM = 'ocean_time'

split_config = ManifestSplittingConfig.from_dict({
    ManifestSplitCondition.AnyArray(): {
        ManifestSplitDimCondition.DimensionName(TIME_DIM): 365 * 24
    }
})
config = icechunk.RepositoryConfig(manifest=ManifestConfig(splitting=split_config))
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f's3://{bucket}/',
        store=icechunk.s3_store(region=region, anonymous=True),
    )
)
creds = icechunk.containers_credentials(
    {f's3://{bucket}/': icechunk.s3_credentials(anonymous=True)}
)
storage = icechunk.s3_storage(bucket=bucket, prefix=prefix, region=region, anonymous=True)
repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=creds)
session = repo.readonly_session('main')
ds = xr.open_zarr(session.store, consolidated=False)
ds

## 2. Python visualization (hvplot.quadmesh)

The same curvilinear grid the JS demo renders with deck.gl.
`rasterize=True` uses datashader under the hood — the JS equivalent
tessellates into ~300K quads and renders directly on the GPU.

In [ ]:
SAMPLE_TIME = '2012-10-29 12:00'   # Hurricane Sandy peak

zeta_2d = ds['zeta'].cf.sel(T=SAMPLE_TIME, method='nearest').load()
print(f'zeta range: {float(zeta_2d.min()):.3f} to {float(zeta_2d.max()):.3f} m')

zeta_2d.hvplot.quadmesh(
    x='lon_rho', y='lat_rho',
    rasterize=True, geo=True, tiles='OSM',
    cmap='bwr', clim=(-1.5, 1.5),
    title=f'Sea Surface Height — {SAMPLE_TIME} (Hurricane Sandy)',
    width=800, height=450,
)

## 3. What the JS tessellation does

Python equivalent of `tessellate()` in `curvilinear_mesh_layer.js`.
Each grid cell becomes a quadrilateral defined by its 4 corner lon/lat points.

In [ ]:
lon_rho = ds['lon_rho'].values
lat_rho = ds['lat_rho'].values
rows, cols = lon_rho.shape
n_cells = (rows - 1) * (cols - 1)

print(f'Grid: {rows} × {cols} = {rows*cols:,} vertices')
print(f'Quads: {n_cells:,} → {n_cells*2:,} triangles')
print(f'Tessellation typed arrays (JS side):')
print(f'  positions  Float32Array  {n_cells*4*2:,} values  ({n_cells*4*2*4/1e6:.1f} MB)')
print(f'  fillColors Uint8Array    {n_cells*4:,} values   ({n_cells*4/1e6:.1f} MB)')
print(f'  startIndices Int32Array  {n_cells+1:,} values')

In [ ]:
# Python equivalent of the tessellate() inner loop
i, j = np.meshgrid(np.arange(rows-1), np.arange(cols-1), indexing='ij')
sw = np.stack([lon_rho[i,   j  ], lat_rho[i,   j  ]], axis=-1)
se = np.stack([lon_rho[i,   j+1], lat_rho[i,   j+1]], axis=-1)
ne = np.stack([lon_rho[i+1, j+1], lat_rho[i+1, j+1]], axis=-1)
nw = np.stack([lon_rho[i+1, j  ], lat_rho[i+1, j  ]], axis=-1)
# quads[cell, vertex(0-3), coord(lon/lat)]
quads = np.stack([sw, se, ne, nw], axis=2)
print(f'quads shape: {quads.shape}  (CCW winding: SW→SE→NE→NW)')

## 4. View the browser demo

Start the Vite dev server from the repo root:

```bash
npm install
npm run dev
```

Then open http://localhost:5173 — or embed it here:

In [ ]:
from IPython.display import IFrame
# Requires `npm run dev` to be running in this directory
IFrame('http://localhost:5173', width=900, height=560)

## 5. How icechunk.js opens the same store in the browser

The JS equivalent of the Python code in Cell 1 above:

```js
// src/main.js (see full source)
import { Repository } from '@earthmover/icechunk';
import { createFetchStorage } from '@earthmover/icechunk/fetch-storage';
import { root, open, get, slice } from 'zarrita';

// createFetchStorage: read-only S3 access via browser fetch() — no AWS credentials
const storage = createFetchStorage(
  'https://usgs-coawst.s3.us-west-2.amazonaws.com' +
  '/useast-archive/icechunk/coawst-useast.icechunk'
);

// authorizeVirtualChunkAccess: icechunk resolves virtual chunk references
// (pointers to byte ranges in the source NetCDF files) via additional fetch() calls
const repo = await Repository.open(storage, undefined, {
  'https://usgs-coawst.s3.us-west-2.amazonaws.com/': null,  // null = anonymous
});

const session = await repo.readonlySession({ branch: 'main' });
const store = session.store;

// Read lon_rho, lat_rho, and one time slice of zeta via zarrita
const r = root(store);
const zetaArr = await open(r.resolve('zeta'), { kind: 'array' });
const { data } = await get(zetaArr, [27000, slice(null), slice(null)]);
// → Float32Array of shape [336, 896]
```